In [1]:
import pandas as pd

# 1. Cargar los archivos
# Asegúrate de que los nombres coincidan con tus archivos locales
df_irene = pd.read_excel('datos_ieq_irene_2023_merge_final.xlsx')
df_avellanos = pd.read_excel('datos_ieq_avellanos_2023_merge_final.xlsx')

# 2. Crear la columna identificadora en cada uno
df_irene['escuela'] = 'Irene Frei'
df_avellanos['escuela'] = 'Los Avellanos'

# 3. Unir las tablas (una debajo de la otra)
# Usamos concat porque tienen las mismas columnas
df_final = pd.concat([df_irene, df_avellanos], ignore_index=True)

# 4. Verificar el resultado
print(df_final['escuela'].value_counts())
# Esto te mostrará cuántos datos hay de cada una en el mismo archivo


escuela
Irene Frei       32456
Los Avellanos     6010
Name: count, dtype: int64


In [2]:
# Definimos una función de estilo para aplicar una paleta de colores verdes (AulaSana)
def style_pastel_green(df):
    return df.head().style.set_table_styles([
        {
            'selector': 'th', 
            'props': [
                ('background-color', '#18B889'), # Verde principal
                ('color', 'white'),              # Texto blanco
                ('font-weight', 'bold'),
                ('text-align', 'center')
            ]
        },
        {
            'selector': 'td', 
            'props': [
                ('background-color', '#F0F9F6'), # Verde menta suave
                ('text-align', 'center')
            ]
        }
    ]).set_properties(**{
        'border': '1px solid #E0E0E0'            # Borde sutil
    }) # <--- Asegúrate de que este paréntesis cierre el .set_properties

In [3]:
# Verificación visual 
print("📊 Dataset Df: Calidad de Aire en Salas de Clases Colegio Irene & Colegio Avellanos")
display(style_pastel_green(df_final))

📊 Dataset Df: Calidad de Aire en Salas de Clases Colegio Irene & Colegio Avellanos


,date,temp_int1,hum_int1,CO2_int1,PM 2.5_int1,PM 10_int1,temp_int2,hum_int2,CO2_int2,PM 2.5_int2,PM 10_int2,temp_ext,hum_ext,CO2_ext,PM 2.5_ext,PM 10_ext,escuela
0,2023-04-20 14:30:00,20.780000,66.210000,1011.350000,12.960000,15.310000,20.850000,66.830000,796.180000,10.270000,20.190000,17.494348,75.076320,406.455829,13.060778,19.088471,Irene Frei
1,2023-04-21 11:00:00,20.720000,70.420000,1346.320000,10.440000,15.060000,20.230000,72.820000,1438.160000,6.140000,10.670000,17.758792,75.302252,420.272067,9.727599,15.397579,Irene Frei
2,2023-04-24 23:20:00,109.490000,68.000000,412.870000,61.410000,67.090000,17.500000,67.700000,949.620000,60.250000,77.790000,59.475170,74.049808,410.898998,66.162385,71.061827,Irene Frei
3,2023-04-28 05:00:00,20.600000,0.810000,396.090000,8.780000,9.570000,17.900000,76.710000,395.050000,6.860000,9.000000,21.077079,47.438806,455.281836,11.553041,10.143717,Irene Frei
4,2023-04-28 08:20:00,22.910000,31.230000,672.520000,5.040000,28.480000,18.320000,76.160000,644.820000,4.680000,9.860000,21.706049,58.045136,444.294087,5.624613,20.399886,Irene Frei


In [4]:
# 1. Aseguramos formato datetime (si no lo has hecho ya)
df_final['date'] = pd.to_datetime(df_final['date'])

# 2. Creamos una columna temporal de año para facilitar la agrupación
df_final['año'] = df_final['date'].dt.year

# 3. Agrupamos por Año y Escuela para contar los registros
conteo_anual_colegio = df_final.groupby(['año', 'escuela']).size().reset_index(name='cantidad_registros')

# 4. Mostramos el resultado
print("Desglose de datos por Año y Colegio:")
print(conteo_anual_colegio)

Desglose de datos por Año y Colegio:
    año        escuela  cantidad_registros
0  2023     Irene Frei                8862
1  2023  Los Avellanos                6010
2  2024     Irene Frei               23576
3  2025     Irene Frei                  18


In [5]:
pip install pywaffle

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [7]:
# =========================
# CONFIGURACIÓN BASE
# =========================
colegio_target = "Los Avellanos"

# Asegurar formato fecha
df_final["date"] = pd.to_datetime(df_final["date"], errors="coerce")

# Período que quieres mostrar
start_date = "2023-03-27 08:00:00"
end_date   = "2023-03-29 00:00:00"

# Filtrar colegio + rango de fechas
df_base = df_final[
    (df_final["escuela"] == colegio_target) &
    (df_final["date"] >= start_date) &
    (df_final["date"] <= end_date)
].copy()

# Ordenar
df_base = df_base.sort_values("date")

# Verificar
print(df_base.shape)
print(df_base[["date", "escuela"]].head())

(229, 18)
                     date        escuela
32765 2023-03-27 09:30:00  Los Avellanos
32766 2023-03-27 10:10:00  Los Avellanos
32767 2023-03-27 10:20:00  Los Avellanos
32768 2023-03-27 10:30:00  Los Avellanos
32769 2023-03-27 10:40:00  Los Avellanos


In [8]:
# =========================
# PREPARAR DATOS BUBBLE
# =========================

import pandas as pd

# Usamos df_base, que ya viene filtrado por colegio y fechas
df_bubble = df_base[[
    "date",
    "temp_int1", "CO2_int1", "hum_int1",
    "temp_int2", "CO2_int2", "hum_int2"
]].copy()

# Convertir a numérico
for col in [
    "temp_int1", "CO2_int1", "hum_int1",
    "temp_int2", "CO2_int2", "hum_int2"
]:
    df_bubble[col] = pd.to_numeric(df_bubble[col], errors="coerce")

# Eliminar filas sin fecha
df_bubble = df_bubble.dropna(subset=["date"])

print(df_bubble.head())


                     date  temp_int1  CO2_int1  hum_int1  temp_int2  CO2_int2  \
32765 2023-03-27 09:30:00      20.50    668.58     44.55      20.30    772.75   
32766 2023-03-27 10:10:00      20.61    714.31     44.36      20.41    686.23   
32767 2023-03-27 10:20:00      20.67    776.82     44.21      20.61    601.00   
32768 2023-03-27 10:30:00      20.75    789.79     45.04      20.85    658.30   
32769 2023-03-27 10:40:00      20.94    809.42     47.77      21.18    777.59   

       hum_int2  
32765     47.42  
32766     47.31  
32767     47.20  
32768     47.60  
32769     49.64  


In [9]:
# =========================
# FORMATO LARGO
# =========================

# Sala 1
df_int1 = df_bubble[["date", "temp_int1", "CO2_int1", "hum_int1"]].copy()
df_int1.columns = ["date", "temp", "CO2", "hum"]
df_int1["sala"] = "Sala 1"

# Sala 2
df_int2 = df_bubble[["date", "temp_int2", "CO2_int2", "hum_int2"]].copy()
df_int2.columns = ["date", "temp", "CO2", "hum"]
df_int2["sala"] = "Sala 2"

# Unir
df_bubble_long = pd.concat([df_int1, df_int2], ignore_index=True)

# Quitar filas con valores faltantes
df_bubble_long = df_bubble_long.dropna(subset=["temp", "CO2", "hum"])

# Crear etiqueta de tiempo para animación
df_bubble_long["frame"] = df_bubble_long["date"].dt.strftime("%Y-%m-%d %H:%M")

print(df_bubble_long.head())
print(df_bubble_long.shape)

                 date   temp     CO2    hum    sala             frame
0 2023-03-27 09:30:00  20.50  668.58  44.55  Sala 1  2023-03-27 09:30
1 2023-03-27 10:10:00  20.61  714.31  44.36  Sala 1  2023-03-27 10:10
2 2023-03-27 10:20:00  20.67  776.82  44.21  Sala 1  2023-03-27 10:20
3 2023-03-27 10:30:00  20.75  789.79  45.04  Sala 1  2023-03-27 10:30
4 2023-03-27 10:40:00  20.94  809.42  47.77  Sala 1  2023-03-27 10:40
(458, 6)


In [10]:
# =========================
# BASE LOS AVELLANOS
# =========================

import pandas as pd

colegio_target = "Los Avellanos"

df_final["date"] = pd.to_datetime(df_final["date"], errors="coerce")

start_date = "2023-03-27 08:00:00"
end_date   = "2023-03-29 00:00:00"

df_base = df_final[
    (df_final["escuela"] == colegio_target) &
    (df_final["date"] >= start_date) &
    (df_final["date"] <= end_date)
].copy()

df_base = df_base.sort_values("date")

print(df_base.shape)
print(df_base[["date", "escuela"]].head())

(229, 18)
                     date        escuela
32765 2023-03-27 09:30:00  Los Avellanos
32766 2023-03-27 10:10:00  Los Avellanos
32767 2023-03-27 10:20:00  Los Avellanos
32768 2023-03-27 10:30:00  Los Avellanos
32769 2023-03-27 10:40:00  Los Avellanos


In [11]:
import sys
!{sys.executable} -m pip install dash plotly -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, html, dcc, Input, Output

# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================

df_final["date"] = pd.to_datetime(df_final["date"], errors="coerce")

COLOR_FONDO = "#18B889"
COLOR_CARD = "#FFFFFF"

COLOR_AZUL = "#003057"
COLOR_DORADO = "#E2D192"
COLOR_GRIS = "#708090"

COLOR_VERDE = "#4CAF50"
COLOR_ROJO = "#E53935"
COLOR_AZUL_CLARO = "#2E5BFF"

TEMP_MIN = 19.4
TEMP_MAX = 27.7
CO2_MAX = 700
HUM_MIN = 30
HUM_MAX = 60

DATE_START = "2023-03-27"
DATE_END = "2023-03-29"

# =========================================================
# HELPERS
# =========================================================

def card_style():
    return {
        "backgroundColor": COLOR_CARD,
        "padding": "18px",
        "borderRadius": "18px",
        "boxShadow": "0 3px 12px rgba(0,0,0,0.18)"
    }

def empty_figure(title="Sin datos para los filtros seleccionados", height=320):
    fig = go.Figure()
    fig.update_layout(
        template="simple_white",
        title={"text": title, "x": 0.5},
        height=height,
        paper_bgcolor="white",
        plot_bgcolor="white"
    )
    return fig

def sala_label(sala):
    return "Sala 1" if sala == "int1" else "Sala 2"

def sala_color(sala):
    return COLOR_AZUL if sala == "int1" else COLOR_DORADO

def get_filtered_df(colegio, date_start, date_end):
    start = pd.to_datetime(date_start)
    end = pd.to_datetime(date_end) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

    df = df_final[df_final["escuela"] == colegio].copy()
    df = df[(df["date"] >= start) & (df["date"] <= end)].copy()
    df = df.sort_values("date")
    return df

def get_selected_salas(modo, sala_principal, sala_comparacion):
    if modo == "comparar":
        if sala_principal == sala_comparacion:
            return [sala_principal]
        return [sala_principal, sala_comparacion]
    return [sala_principal]

def resample_for_plot(df, freq):
    tmp = df.copy()
    for c in tmp.columns:
        if c != "date":
            tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
    tmp = (
        tmp.set_index("date")
        .resample(freq)
        .mean()
        .reset_index()
    )
    return tmp

# =========================================================
# SERIES TEMPORALES INTERACTIVAS
# =========================================================

def make_time_series(df, variable="temp", modo="una", sala_principal="int1", sala_comparacion="int2"):
    if df.empty:
        return empty_figure(f"{variable.upper()} - Sin datos", height=380)

    plot_df = df.copy().sort_values("date")
    n_days = max(1, (plot_df["date"].max() - plot_df["date"].min()).days + 1)
    freq = "15min" if n_days <= 3 else ("1h" if n_days <= 10 else "1D")
    plot_df = resample_for_plot(plot_df, freq)

    selected = get_selected_salas(modo, sala_principal, sala_comparacion)

    fig = go.Figure()

    if variable == "temp":
        if "int1" in selected:
            fig.add_trace(go.Scatter(
                x=plot_df["date"], y=plot_df["temp_int1"],
                mode="lines", name="Temp sala 1",
                line=dict(color="#1f77b4", width=2.2)
            ))
        if "int2" in selected:
            fig.add_trace(go.Scatter(
                x=plot_df["date"], y=plot_df["temp_int2"],
                mode="lines", name="Temp sala 2",
                line=dict(color="#ff7f0e", width=2.2)
            ))
        fig.add_trace(go.Scatter(
            x=plot_df["date"], y=plot_df["temp_ext"],
            mode="lines", name="Temp exterior",
            line=dict(color="#2ca02c", width=2.0)
        ))

        fig.add_hrect(
            y0=TEMP_MIN, y1=TEMP_MAX,
            fillcolor="rgba(76,175,80,0.10)",
            line_width=0,
            layer="below"
        )
        fig.add_hline(y=TEMP_MIN, line_dash="dash", line_color="blue", line_width=2)
        fig.add_hline(y=TEMP_MAX, line_dash="dash", line_color="red", line_width=2)

        title = "Evolución temporal de temperatura en salas de clases"
        ytitle = "Temperatura (°C)"

    elif variable == "CO2":
        if "int1" in selected:
            fig.add_trace(go.Scatter(
                x=plot_df["date"], y=plot_df["CO2_int1"],
                mode="lines", name="CO₂ sala 1",
                line=dict(color="#1f77b4", width=2.2)
            ))
        if "int2" in selected:
            fig.add_trace(go.Scatter(
                x=plot_df["date"], y=plot_df["CO2_int2"],
                mode="lines", name="CO₂ sala 2",
                line=dict(color="#ff7f0e", width=2.2)
            ))
        fig.add_trace(go.Scatter(
            x=plot_df["date"], y=plot_df["CO2_ext"],
            mode="lines", name="CO₂ exterior",
            line=dict(color="#2ca02c", width=2.0)
        ))

        fig.add_hrect(
            y0=0, y1=CO2_MAX,
            fillcolor="rgba(76,175,80,0.08)",
            line_width=0,
            layer="below"
        )
        fig.add_hline(y=CO2_MAX, line_dash="dash", line_color="red", line_width=2)

        title = "Evolución temporal de CO₂ en salas de clases"
        ytitle = "CO₂ (ppm)"

    else:
        if "int1" in selected:
            fig.add_trace(go.Scatter(
                x=plot_df["date"], y=plot_df["hum_int1"],
                mode="lines", name="Humedad sala 1",
                line=dict(color=COLOR_AZUL, width=2.2)
            ))
        if "int2" in selected:
            fig.add_trace(go.Scatter(
                x=plot_df["date"], y=plot_df["hum_int2"],
                mode="lines", name="Humedad sala 2",
                line=dict(color=COLOR_DORADO, width=2.2)
            ))
        fig.add_trace(go.Scatter(
            x=plot_df["date"], y=plot_df["hum_ext"],
            mode="lines", name="Humedad exterior",
            line=dict(color=COLOR_GRIS, width=2.0)
        ))

        fig.add_hrect(
            y0=HUM_MIN, y1=HUM_MAX,
            fillcolor="rgba(76,175,80,0.10)",
            line_width=0,
            layer="below"
        )
        fig.add_hline(y=HUM_MIN, line_dash="dash", line_color="blue", line_width=2)
        fig.add_hline(y=HUM_MAX, line_dash="dash", line_color="red", line_width=2)

        title = "Evolución temporal de humedad relativa en salas de clases"
        ytitle = "Humedad relativa (%)"

    fig.update_layout(
        template="simple_white",
        title={"text": title, "x": 0.5},
        height=380,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=60, r=25, t=55, b=90),
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.28,
            xanchor="center",
            x=0.5
        ),
        hovermode="x unified"
    )
    fig.update_xaxes(
        title="Fecha (hh:mm dd/mm/yy)",
        showgrid=True,
        gridcolor="rgba(0,0,0,0.06)",
        tickformat="%H:%M - %d/%m/%y"
    )
    fig.update_yaxes(
        title=ytitle,
        showgrid=True,
        gridcolor="rgba(0,0,0,0.06)"
    )
    return fig

# =========================================================
# BUBBLE CHART
# =========================================================

def make_bubble(df):
    if df.empty:
        return empty_figure("Bubble chart de correlación - Sin datos", height=540)

    cols = ["date", "temp_int1", "CO2_int1", "hum_int1", "temp_int2", "CO2_int2", "hum_int2"]
    tmp = df[cols].copy()

    # convertir a numérico
    for c in cols:
        if c != "date":
            tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

    # resample
    tmp = tmp.resample("1h", on="date").mean().reset_index().dropna()

    if tmp.empty:
        return empty_figure("Bubble chart de correlación - Sin datos", height=540)

    # reshape salas
    df1 = tmp[["date", "temp_int1", "CO2_int1", "hum_int1"]].copy()
    df1.columns = ["date", "temp", "CO2", "hum"]
    df1["sala"] = "Sala 1"

    df2 = tmp[["date", "temp_int2", "CO2_int2", "hum_int2"]].copy()
    df2.columns = ["date", "temp", "CO2", "hum"]
    df2["sala"] = "Sala 2"

    long = pd.concat([df1, df2], ignore_index=True).dropna()
    long["frame"] = long["date"].dt.strftime("%Y-%m-%d %H:%M")
    long["hum_size"] = long["hum"].clip(lower=20, upper=80)

    fig = go.Figure()

    # frames (animación)
    frames = []
    for frame_name in long["frame"].unique():
        dff = long[long["frame"] == frame_name]

        frames.append(go.Frame(
            name=frame_name,
            data=[go.Scatter(
                x=dff["temp"],
                y=dff["CO2"],
                mode="markers+text",
                text=dff["sala"],
                textposition="top center",
                marker=dict(
                    size=dff["hum_size"] * 0.75,
                    color=dff["sala"].map({"Sala 1": COLOR_AZUL, "Sala 2": COLOR_DORADO}),
                    line=dict(color="white", width=1.6),
                    opacity=0.85
                ),
                showlegend=False
            )]
        ))

    # primer frame visible
    first = long[long["frame"] == long["frame"].iloc[0]]

    fig.add_trace(go.Scatter(
        x=first["temp"],
        y=first["CO2"],
        mode="markers+text",
        text=first["sala"],
        textposition="top center",
        marker=dict(
            size=first["hum_size"] * 0.75,
            color=first["sala"].map({"Sala 1": COLOR_AZUL, "Sala 2": COLOR_DORADO}),
            line=dict(color="white", width=1.6),
            opacity=0.85
        ),
        showlegend=False
    ))

    fig.frames = frames

    # zonas de confort
    fig.add_vrect(
        x0=TEMP_MIN, x1=TEMP_MAX,
        fillcolor="rgba(76,175,80,0.08)",
        line_width=0,
        layer="below"
    )

    fig.add_vline(x=TEMP_MIN, line_dash="dash", line_color=COLOR_AZUL_CLARO, line_width=2.2)
    fig.add_vline(x=TEMP_MAX, line_dash="dash", line_color=COLOR_ROJO, line_width=2.2)
    fig.add_hline(y=CO2_MAX, line_dash="dash", line_color=COLOR_ROJO, line_width=2.2)

    # layout
    fig.update_layout(
        template="simple_white",
        title={"text": "Bubble chart de correlación", "x": 0.5},
        height=540,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=55, r=20, t=55, b=260),

        # botón play
        updatemenus=[{
            "type": "buttons",
            "showactive": False,
            "x": 0.02,
            "y": -0.22,
            "xanchor": "left",
            "yanchor": "top",
            "buttons": [{
                "label": "▶ Play",
                "method": "animate",
                "args": [None, {
                    "frame": {"duration": 500, "redraw": True},
                    "transition": {"duration": 250},
                    "fromcurrent": True
                }]
            }]
        }],

        # anotaciones
        annotations=[
            dict(
                x=TEMP_MIN,
                y=1.04,
                xref="x",
                yref="paper",
                text="Temp mín. confort",
                showarrow=False,
                font=dict(size=10, color=COLOR_AZUL_CLARO)
            ),
            dict(
                x=TEMP_MAX,
                y=1.04,
                xref="x",
                yref="paper",
                text="Temp máx. confort",
                showarrow=False,
                font=dict(size=10, color=COLOR_ROJO)
            ),
            dict(
                x=0.5,
                y=-0.48,
                xref="paper",
                yref="paper",
                showarrow=False,
                text=(
                    "Este gráfico permite analizar de forma integrada la temperatura "
                    "(eje X), CO₂ (eje Y) y humedad relativa (tamaño de burbuja).<br>"
                    "Cada color representa una sala distinta; las líneas indican "
                    "rangos de confort térmico y el límite recomendado de CO₂."
                ),
                font=dict(size=14, color="#444"),
                xanchor="center",
                align="center"
            )
        ]
    )

    # ejes
    fig.update_xaxes(
        title="Temperatura (°C)",
        range=[14, 35],
        showgrid=True,
        gridcolor="rgba(0,0,0,0.06)"
    )

    fig.update_yaxes(
        title="CO₂ (ppm)",
        range=[0, max(900, long["CO2"].max() + 100)],
        showgrid=True,
        gridcolor="rgba(0,0,0,0.06)"
    )

    return fig
    
# =========================================================
# BOXPLOT
# =========================================================

def make_boxplot(df):
    if df.empty:
        return empty_figure("Boxplot de temperatura - Sin datos", height=340)

    data = []
    for s in ["int1", "int2"]:
        col = f"temp_{s}"
        vals = pd.to_numeric(df[col], errors="coerce").dropna()
        data.append((sala_label(s), vals, sala_color(s)))

    vals_ext = pd.to_numeric(df["temp_ext"], errors="coerce").dropna()
    data.append(("Exterior", vals_ext, COLOR_GRIS))

    fig = go.Figure()
    for label, vals, color in data:
        fig.add_trace(go.Box(y=vals, name=label, marker_color=color, boxmean=True))

    fig.add_hrect(y0=TEMP_MIN, y1=TEMP_MAX, fillcolor="rgba(76,175,80,0.08)", line_width=0, layer="below")

    fig.update_layout(
        template="simple_white",
        title={"text": "Boxplot de temperatura", "x": 0.5},
        height=340,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=45, r=20, t=55, b=35),
        showlegend=False
    )
    fig.update_yaxes(title="Temperatura (°C)")
    return fig

# =========================================================
# HISTOGRAMA
# =========================================================

def make_histogram(df):
    if df.empty:
        return empty_figure("Histograma comparativo - Sin datos", height=340)

    s1 = pd.to_numeric(df["temp_int1"], errors="coerce").dropna()
    s2 = pd.to_numeric(df["temp_int2"], errors="coerce").dropna()

    fig = go.Figure()
    fig.add_trace(go.Histogram(x=s1, name="Sala 1", opacity=0.62, nbinsx=16, marker_color=COLOR_AZUL))
    fig.add_trace(go.Histogram(x=s2, name="Sala 2", opacity=0.55, nbinsx=16, marker_color=COLOR_DORADO))

    fig.add_vrect(x0=TEMP_MIN, x1=TEMP_MAX, fillcolor="rgba(76,175,80,0.08)", line_width=0, layer="below")
    fig.add_vline(x=TEMP_MIN, line_dash="dash", line_color=COLOR_AZUL_CLARO, line_width=2)
    fig.add_vline(x=TEMP_MAX, line_dash="dash", line_color=COLOR_ROJO, line_width=2)

    fig.update_layout(
        template="simple_white",
        title={"text": "Histograma comparativo Sala 1 vs Sala 2", "x": 0.5},
        height=340,
        paper_bgcolor="white",
        plot_bgcolor="white",
        barmode="overlay",
        margin=dict(l=45, r=20, t=55, b=35),
        legend=dict(orientation="h", y=1.02, x=1, xanchor="right")
    )
    fig.update_xaxes(title="Temperatura (°C)")
    fig.update_yaxes(title="Frecuencia")
    return fig

# =========================================================
# PANEL JAVI ANIMADO
# =========================================================

def make_quality_panel_animated(df):
    if df.empty:
        return empty_figure("Panel de calidad ambiental - Sin datos", height=760)

    cols = ["date", "temp_int1", "temp_int2", "temp_ext"]
    tmp = df[cols].copy()

    for c in ["temp_int1", "temp_int2", "temp_ext"]:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

    tmp = tmp.dropna()
    if tmp.empty:
        return empty_figure("Panel de calidad ambiental - Sin datos", height=760)

    tmp = (
        tmp.resample("1h", on="date")
        .mean()
        .reset_index()
        .dropna()
    )

    def score_temp(v):
        if pd.isna(v):
            return 0
        if TEMP_MIN <= v <= TEMP_MAX:
            return 100
        if v < TEMP_MIN:
            diff = TEMP_MIN - v
        else:
            diff = v - TEMP_MAX
        return max(0, round(100 - diff * 20, 1))

    tmp["score_int1"] = tmp["temp_int1"].apply(score_temp)
    tmp["score_int2"] = tmp["temp_int2"].apply(score_temp)
    tmp["score_ext"] = tmp["temp_ext"].apply(score_temp)
    tmp["frame"] = tmp["date"].dt.strftime("%Y-%m-%d %H:%M")

    dias_es = {
        "Monday": "Lunes",
        "Tuesday": "Martes",
        "Wednesday": "Miércoles",
        "Thursday": "Jueves",
        "Friday": "Viernes",
        "Saturday": "Sábado",
        "Sunday": "Domingo"
    }

    def fecha_es(dt):
        dia = dias_es.get(dt.strftime("%A"), dt.strftime("%A"))
        return f"{dia}, {dt.strftime('%d-%m-%Y')} | {dt.strftime('%H:%M')} hrs"

    def build_panel_row(row):
        fig = make_subplots(
            rows=2,
            cols=3,
            row_heights=[0.48, 0.52],
            vertical_spacing=0.22,
            horizontal_spacing=0.12,
            specs=[
                [{"type": "scatter"}, {"type": "scatter"}, {"type": "scatter"}],
                [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]
            ],
            subplot_titles=(
                "Sala 1", "Sala 2", "Exterior",
                "Confort térmico Sala 1", "Confort térmico Sala 2", "Temperatura Exterior"
            )
        )

        temps = [row["temp_int1"], row["temp_int2"], row["temp_ext"]]
        labels = ["Sala 1", "Sala 2", "Exterior"]
        colors = [COLOR_AZUL, COLOR_DORADO, COLOR_GRIS]
        scores = [row["score_int1"], row["score_int2"], row["score_ext"]]

        # -------- FILA SUPERIOR: temperatura puntual --------
        for i, (lab, temp, col) in enumerate(zip(labels, temps, colors), start=1):
            fig.add_hrect(
                y0=TEMP_MIN,
                y1=TEMP_MAX,
                fillcolor="rgba(76,175,80,0.10)",
                line_width=0,
                row=1,
                col=i
            )

            fig.add_hline(
                y=TEMP_MIN,
                line_dash="dash",
                line_color=COLOR_AZUL_CLARO,
                line_width=1.8,
                row=1,
                col=i
            )
            fig.add_hline(
                y=TEMP_MAX,
                line_dash="dash",
                line_color=COLOR_ROJO,
                line_width=1.8,
                row=1,
                col=i
            )

            fig.add_trace(
                go.Scatter(
                    x=[lab],
                    y=[temp],
                    mode="markers+text",
                    text=[f"{temp:.1f}°C"],
                    textposition="top center",
                    marker=dict(
                        size=26,
                        color=col,
                        line=dict(color="white", width=2.2)
                    ),
                    showlegend=False
                ),
                row=1,
                col=i
            )

            fig.update_xaxes(
                title_text="Espacio evaluado",
                showticklabels=True,
                tickfont=dict(size=11),
                title_font=dict(size=12),
                row=1,
                col=i
            )

            fig.update_yaxes(
                title_text="Temperatura (°C)",
                range=[8, 35],
                tickfont=dict(size=11),
                title_font=dict(size=12),
                gridcolor="rgba(0,0,0,0.08)",
                zeroline=False,
                row=1,
                col=i
            )

        # -------- FILA INFERIOR: gauges --------
        for i, (val, col) in enumerate(zip(scores, colors), start=1):
            fig.add_trace(
                go.Indicator(
                    mode="gauge+number",
                    value=val,
                    number={
                        "suffix": "%",
                        "font": {"size": 20, "color": "#222"}
                    },
                    gauge={
                        "shape": "angular",
                        "axis": {
                            "range": [0, 100],
                            "tickwidth": 1,
                            "tickfont": {"size": 10}
                        },
                        "bar": {
                            "color": col,
                            "thickness": 0.28
                        },
                        "bgcolor": "white",
                        "borderwidth": 0,
                        "steps": [
                            {"range": [0, 33], "color": "#FDECEC"},
                            {"range": [33, 67], "color": "#FFF4DA"},
                            {"range": [67, 100], "color": "#EAF6EA"},
                        ]
                    }
                ),
                row=2,
                col=i
            )

        fig.update_annotations(font=dict(size=19, color="#222"))

        fig.update_layout(
            template="simple_white",
            height=820,
            paper_bgcolor="white",
            plot_bgcolor="white",
            margin=dict(l=55, r=40, t=120, b=100),
            title={
                "text": (
                    "<span style='font-size:28px'><b>AulaSana: Los Avellanos</b></span><br>"
                    "<span style='font-size:20px; color:#333'><b>Dinámica escolar</b></span><br>"
                    f"<span style='font-size:17px; color:{COLOR_AZUL}'><b>{fecha_es(row['date'])}</b></span>"
                ),
                "x": 0.5,
                "y": 0.97,
                "xanchor": "center",
                "yanchor": "top"
            },
            updatemenus=[{
                "type": "buttons",
                "showactive": False,
                "x": 0.0,
                "y": -0.08,
                "xanchor": "left",
                "yanchor": "top",
                "buttons": [{
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [None, {
                        "frame": {"duration": 550, "redraw": True},
                        "transition": {"duration": 250},
                        "fromcurrent": True
                    }]
                }]
            }],
            annotations=list(fig.layout.annotations) + [
                dict(
                    x=0.5,
                    y=-0.10,
                    xref="paper",
                    yref="paper",
                    text=(
                        "La franja verde indica el rango de confort térmico; "
                        "los indicadores inferiores resumen el desempeño térmico de cada espacio."
                    ),
                    showarrow=False,
                    font=dict(size=12, color="#555"),
                    xanchor="center",
                    align="center"
                )
            ]
        )

        return fig

    base_fig = build_panel_row(tmp.iloc[0])

    frames = []
    for _, row in tmp.iterrows():
        fr = build_panel_row(row)
        frames.append(
            go.Frame(
                name=row["frame"],
                data=fr.data,
                layout=fr.layout
            )
        )

    base_fig.frames = frames
    return base_fig
    
# =========================================================
# APP
# =========================================================

app = Dash(__name__)

app.layout = html.Div([

    # HEADER
    html.Div([

        html.Div([
            # Logo izquierdo
            html.Div([
                html.Img(
                    src="/assets/logo_proyecto.png",
                    style={
                        "height": "70px",
                        "objectFit": "contain"
                    }
                )
            ], style={
                "flex": "1",
                "display": "flex",
                "justifyContent": "flex-start",
                "alignItems": "center"
            }),

            # Centro
            html.Div([
                html.H1("AulaSana", style={
                    "textAlign": "center",
                    "color": "#111",
                    "marginBottom": "6px",
                    "marginTop": "0px",
                    "fontSize": "56px",
                    "fontWeight": "700"
                }),

                html.P("Diagnóstico de calidad ambiental interior en establecimientos educacionales", style={
                    "textAlign": "center",
                    "color": "#666",
                    "marginTop": "0px",
                    "marginBottom": "0px",
                    "fontSize": "16px"
                })
            ], style={
                "flex": "2",
                "display": "flex",
                "flexDirection": "column",
                "justifyContent": "center",
                "alignItems": "center"
            }),

            # Logos derechos
            html.Div([
                html.Img(
                    src="/assets/logo_udec.png",
                    style={
                        "height": "58px",
                        "objectFit": "contain",
                        "marginRight": "14px"
                    }
                ),
                html.Img(
                    src="/assets/logo_anid.png",
                    style={
                        "height": "58px",
                        "objectFit": "contain"
                    }
                )
            ], style={
                "flex": "1",
                "display": "flex",
                "justifyContent": "flex-end",
                "alignItems": "center"
            })

        ], style={
            "display": "flex",
            "alignItems": "center",
            "justifyContent": "space-between",
            "gap": "20px"
        })

    ], style={
        **card_style(),
        "marginBottom": "20px",
        "padding": "24px 28px"
    }),

    # FILTROS
    html.Div([
        html.Div([
            html.Label("Seleccionar establecimiento", style={"fontWeight": "bold", "marginBottom": "6px"}),
            dcc.Dropdown(
                id="dd_colegio",
                options=[{"label": "Los Avellanos", "value": "Los Avellanos"}],
                value="Los Avellanos",
                clearable=False
            )
        ], style={"flex": "1"}),

        html.Div([
            html.Label("Analizar desde", style={"fontWeight": "bold", "marginBottom": "6px"}),
            dcc.DatePickerSingle(
                id="dp_start",
                date=DATE_START,
                display_format="YYYY-MM-DD"
            )
        ], style={"flex": "1"}),

        html.Div([
            html.Label("Hasta", style={"fontWeight": "bold", "marginBottom": "6px"}),
            dcc.DatePickerSingle(
                id="dp_end",
                date=DATE_END,
                display_format="YYYY-MM-DD"
            )
        ], style={"flex": "1"}),

        html.Div([
            html.Label("Visualización", style={"fontWeight": "bold", "marginBottom": "6px"}),
            dcc.RadioItems(
                id="ri_modo",
                options=[
                    {"label": "Ver una sala", "value": "una"},
                    {"label": "Comparar salas", "value": "comparar"},
                ],
                value="comparar",
                inline=False,
                inputStyle={"marginRight": "6px", "marginLeft": "0px"}
            )
        ], style={"flex": "1"}),

        html.Div([
            html.Label("Seleccionar sala", style={"fontWeight": "bold", "marginBottom": "6px"}),
            dcc.Dropdown(
                id="dd_sala",
                options=[
                    {"label": "Sala 1", "value": "int1"},
                    {"label": "Sala 2", "value": "int2"},
                ],
                value="int1",
                clearable=False
            )
        ], style={"flex": "1"}),

        html.Div([
            html.Label("Comparar con", style={"fontWeight": "bold", "marginBottom": "6px"}),
            dcc.Dropdown(
                id="dd_compare",
                options=[
                    {"label": "Sala 1", "value": "int1"},
                    {"label": "Sala 2", "value": "int2"},
                ],
                value="int2",
                clearable=False
            )
        ], style={"flex": "1"})
    ], style={
        **card_style(),
        "display": "flex",
        "gap": "16px",
        "marginBottom": "20px"
    }),

    # SERIES
    html.Div([
        html.Div([
            dcc.Graph(id="graph_temp")
        ], style={"flex": "1", **card_style()}),

        html.Div([
            dcc.Graph(id="graph_co2")
        ], style={"flex": "1", **card_style()}),
    ], style={
        "display": "flex",
        "gap": "16px",
        "marginBottom": "20px"
    }),

    html.Div([
        dcc.Graph(id="graph_hum")
    ], style={**card_style(), "marginBottom": "20px"}),

    html.Div([
        dcc.Graph(id="graph_bubble")
    ], style={**card_style(), "marginBottom": "20px"}),

    html.Div([
        html.Div([dcc.Graph(id="graph_boxplot")], style={"flex": "1", **card_style()}),
        html.Div([dcc.Graph(id="graph_hist")], style={"flex": "1", **card_style()})
    ], style={
        "display": "flex",
        "gap": "16px",
        "marginBottom": "20px"
    }),

    html.Div([
        dcc.Graph(id="graph_quality_panel")
    ], style={**card_style(), "marginBottom": "20px"}),
    
# BLOQUE FINAL / FOOTER PROYECTO
  html.Div([

        html.P([
            "Este trabajo fue desarrollado en el contexto del proyecto ",
            html.B("Fondecyt de Iniciación N°1240683, "),
            "titulado ",
            html.B("Classroom Retrofit: Evaluating the Potential of Transitioning Existing Schools Towards Healthy and High-Performance Environments as Urgent Response to a Changing Climate"),
            ", liderado por la Dra. Andrea Martinez de la Universidad de Concepción."
        ], style={
            "textAlign": "left",
            "color": "#444",
            "fontSize": "15px",
            "lineHeight": "1.6",
            "maxWidth": "900px",
            "margin": "0 auto 18px auto"
        }),

        html.Div([
            html.Img(
                src="/assets/logo_udec.png",
                style={"height": "58px", "marginRight": "20px"}
            ),
            html.Img(
                src="/assets/logo_anid.png",
                style={"height": "58px"}
            )
        ], style={
            "display": "flex",
            "justifyContent": "center",
            "alignItems": "center"
        })

    ], style={
        **card_style(),
        "marginTop": "20px",
        "padding": "24px 28px"
    })

], style={  # 👈 ESTE ES EL CIERRE FINAL (MUY IMPORTANTE)
    "maxWidth": "1500px",
    "margin": "0 auto",
    "padding": "20px",
    "backgroundColor": COLOR_FONDO,
    "fontFamily": "Arial, sans-serif",
    "minHeight": "100vh"
})
# =========================================================
# CALLBACK
# =========================================================

@app.callback(
    Output("graph_temp", "figure"),
    Output("graph_co2", "figure"),
    Output("graph_hum", "figure"),
    Output("graph_bubble", "figure"),
    Output("graph_boxplot", "figure"),
    Output("graph_hist", "figure"),
    Output("graph_quality_panel", "figure"),
    Input("dd_colegio", "value"),
    Input("dp_start", "date"),
    Input("dp_end", "date"),
    Input("ri_modo", "value"),
    Input("dd_sala", "value"),
    Input("dd_compare", "value"),
)
def update_dashboard(colegio, date_start, date_end, modo, sala, compare_with):
    df = get_filtered_df(colegio, date_start, date_end)

    fig_temp = make_time_series(df, "temp", modo, sala, compare_with)
    fig_co2 = make_time_series(df, "CO2", modo, sala, compare_with)
    fig_hum = make_time_series(df, "hum", modo, sala, compare_with)
    fig_bubble = make_bubble(df)
    fig_box = make_boxplot(df)
    fig_hist = make_histogram(df)
    fig_panel = make_quality_panel_animated(df)

    return fig_temp, fig_co2, fig_hum, fig_bubble, fig_box, fig_hist, fig_panel



In [ ]:
app.run(debug=True, port=8890, use_reloader=False)